# Score projects with the registered model (frozen inference)

Loads the REGISTERED classifier from the MLflow model registry and scores
projects WITHOUT retraining. This is the "Option B" workflow: the model is
frozen, so scoring new/unseen projects is genuine held-out validation, and
scoring active projects gives a live risk score for reporting.

Writes project_risk_scores (Delta) for the Power BI over-cost page.

WHEN TO RUN:
  - To generate a new batch: run 03 -> 04 -> 05 (rebuild features), then THIS (07).
    Do NOT run 06 (that would retrain and defeat the frozen-model validation).
  - Anytime you want refreshed risk scores for active projects.

WHY THIS IS SEPARATE FROM TRAINING:
  Training (06) fits a new model. Scoring (07) reuses the fixed, registered
  model. Keeping them separate means new data is scored against an unchanged
  model -> the new rows are a true holdout, and predicted probabilities stay
  comparable across batches.

#### Cell 1

In [1]:
# Load the registered model (no training here)
import mlflow, mlflow.sklearn
import numpy as np, pandas as pd
from pyspark.sql import functions as F

MODEL_NAME = "construction_overrun_classifier"

# load the latest registered version. (Pin a specific version with
# f"models:/{MODEL_NAME}/3" for strict reproducibility if you like.)
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/latest")
print(f"loaded registered model: {MODEL_NAME} (latest)")


StatementMeta(, 0ad09f49-e0e1-4677-8421-0fb62d21227b, 3, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


loaded registered model: construction_overrun_classifier (latest)


StatementMeta(, 0ad09f49-e0e1-4677-8421-0fb62d21227b, 9, Finished, Available, Finished, False)

#### Cell 2

In [2]:
# Pull the projects to score (all of them; feature cols must match training)
pdf = spark.table("ml_project_classification").toPandas()

FEATURES_CAT = ["project_type", "delivery_method", "region"]
FEATURES_NUM = ["log_contract_value", "square_footage", "pct_budget_mep",
                "planned_duration_days", "is_winter_start"]
FEATURES = FEATURES_CAT + FEATURES_NUM

# same light prep as training (must mirror 06 so inputs match what the model expects)
pdf["is_winter_start"] = pdf["is_winter_start"].astype("Int64").fillna(0).astype(int)
for col in ["square_footage", "planned_duration_days", "pct_budget_mep", "log_contract_value"]:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")
    pdf[col] = pdf[col].fillna(pdf[col].median())
score_df = pdf.dropna(subset=FEATURES_CAT).copy()
print(f"scoring {len(score_df)} projects")

StatementMeta(, 0ad09f49-e0e1-4677-8421-0fb62d21227b, 4, Finished, Available, Finished, False)

scoring 229 projects


#### Cell 3

In [3]:
# Score: predicted probability + risk band (NO retraining)
score_df["risk_probability"] = model.predict_proba(score_df[FEATURES])[:, 1]
score_df["predicted_overrun"] = (score_df["risk_probability"] > 0.5).astype(int)

# band the probability into an actionable risk level for the dashboard
score_df["risk_band"] = pd.cut(
    score_df["risk_probability"],
    bins=[-0.01, 0.33, 0.66, 1.01],
    labels=["Low", "Medium", "High"])

print("Risk band distribution:")
print(score_df["risk_band"].value_counts().sort_index().to_string())


StatementMeta(, 0ad09f49-e0e1-4677-8421-0fb62d21227b, 5, Finished, Available, Finished, False)

Risk band distribution:
risk_band
Low       77
Medium    58
High      94


#### Cell 4

In [4]:
# Assemble the output table (predictions + context for reporting)
# For COMPLETED projects we also have the actual outcome -> enables predicted-vs-
# actual validation on the report. For ACTIVE projects the score is a live flag.
out = score_df[[
    "project_id", "project_sk", "project_type", "delivery_method", "region",
    "status", "data_split",
    "risk_probability", "predicted_overrun", "risk_band",
    "is_overrun", "project_overrun",    # actual outcome (null-ish/for-reference on active)
]].copy()

# flag whether the model was right, for completed projects (validation aid)
out["prediction_correct"] = np.where(
    out["status"].isin(["Complete", "Closed"]),
    (out["predicted_overrun"] == out["is_overrun"]),
    None)

sdf_out = spark.createDataFrame(out)


StatementMeta(, 0ad09f49-e0e1-4677-8421-0fb62d21227b, 6, Finished, Available, Finished, False)

#### Cell 5

In [5]:
# Write project_risk_scores (Delta) for Power BI
(sdf_out.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("project_risk_scores"))
print(f"project_risk_scores written: {sdf_out.count()} rows")


StatementMeta(, 0ad09f49-e0e1-4677-8421-0fb62d21227b, 7, Finished, Available, Finished, False)

project_risk_scores written: 229 rows


#### Cell 6

In [6]:
# Validation summary: how did the frozen model do on the test split?
from pyspark.sql import functions as F
print("=== FROZEN-MODEL VALIDATION ===\n")

# accuracy on completed projects, by split (test = genuinely unseen if provenance)
scored = spark.table("project_risk_scores").filter(F.col("prediction_correct").isNotNull())
by_split = (scored.groupBy("data_split")
    .agg(F.round(F.mean(F.col("prediction_correct").cast("double")), 3).alias("accuracy"),
         F.count("*").alias("n")))
by_split.show()

print("Mean risk probability by ACTUAL outcome (should be higher for true over-cost):")
(scored.groupBy("is_overrun")
    .agg(F.round(F.mean("risk_probability"), 3).alias("avg_risk_prob"),
         F.count("*").alias("n"))
    .show())

print("Risk band vs actual over-cost rate (calibration sanity -- higher band -> higher rate):")
(scored.groupBy("risk_band")
    .agg(F.round(F.mean(F.col("is_overrun").cast("double")), 3).alias("actual_overrun_rate"),
         F.count("*").alias("n"))
    .orderBy("risk_band")
    .show())


StatementMeta(, 0ad09f49-e0e1-4677-8421-0fb62d21227b, 8, Finished, Available, Finished, False)

=== FROZEN-MODEL VALIDATION ===

+----------+--------+---+
|data_split|accuracy|  n|
+----------+--------+---+
|     train|   0.867| 60|
|      test|     0.8| 70|
+----------+--------+---+

Mean risk probability by ACTUAL outcome (should be higher for true over-cost):
+----------+-------------+---+
|is_overrun|avg_risk_prob|  n|
+----------+-------------+---+
|         1|        0.747| 71|
|         0|        0.351| 59|
+----------+-------------+---+

Risk band vs actual over-cost rate (calibration sanity -- higher band -> higher rate):
+---------+-------------------+---+
|risk_band|actual_overrun_rate|  n|
+---------+-------------------+---+
|     High|              0.836| 61|
|      Low|              0.086| 35|
|   Medium|                0.5| 34|
+---------+-------------------+---+

